# E0 data checks — Jaden

**The shared `e0_full` folder has now been inspected read-only. Metadata and rollout checks are complete; full numerical activation verification remains open.**

I used the actual metadata, configuration, result tables and logs from the shared folder. Nothing in Google Drive was edited, renamed, deleted or uploaded. This notebook and the repository handoff are separate copies.

My scope is trace cleaning, BOS consistency and prefix alignment. No language-model generation, new continuation sampling, research-data probe retraining or error-injection experiment was run. CPU tests use small synthetic fixtures. The original repository reference is commit `732da15decc8433226210a3e835f618c57db3543`.

The default cells only display saved findings. **Jaden_Repo_Fixes.zip** contains the code and audit evidence. **Jaden_E0_Handoff.zip** contains the previously cleaned rollouts and pinned tokenizer.

## What I verified from the actual files

| Check | Result |
|---|---|
| Source data | 2,202 unique traces from 367 problems |
| New full metadata | 15,380 rows, consistent IDs, labels, source lines and row order |
| Fixed-position prefix text | All 13,219 rows match the pinned-tokenizer reconstruction |
| End-of-trace text | All 2,161 rows match their source traces |
| Layer configuration | 29 entries: embedding plus layers 0–27 |
| Cleaned rollouts | 1,430 complete records / 11,440 continuations |
| Saved rollout hashes | All 1,430 match reconstructed two-BOS inputs |
| Main-run overlap | 794 prefix records agree with the full metadata's declared two-BOS inputs |
| Pilot overlap | 549 prefix records from 183 traces disagree: metadata declares one BOS, rollouts hash to two |
| Missing rollout | `ceil/216_2/p4096`; the latest table explicitly excludes it at 4096 |
| Saved result tables | All 805 long-form result cells agree with the rounded summary tables |

The latest recovery-comparison table already **excludes the pilot**, so the pilot mismatch is not an additional error in that main-only comparison. Its cohort is 265 traces / 52 problems at 512 and 2048, and 264 traces / 52 problems at 4096. Do not mix it with the older 448-trace / 87-problem comparison or the larger grid table.

The metadata agrees internally and its declared main-run input reconstruction matches the rollout hashes. This does **not** prove which inputs produced the numerical activations: original generation IDs and extraction-time input fingerprints were not stored.

Earlier checks remain valid: the nine-layer legacy array was finite; all 11,409 legacy prefixes matched reconstructed text. Ten corrupt opening lines were quarantined from the rollout file; 16 recovery-source traces retain EOS-aware diagnostic count flags; these flags alone do not establish original token identity.

## What I changed in the repository

- Future generation and rollout records save their actual token IDs with hashes.
- Extraction prefers validated saved IDs. Legacy BOS choices require an explicit policy to override the old default and remain labeled as reconstructed.
- Cache checks bind ordered inputs, metadata and activation files. Approximate cosine checks can no longer certify an old cache through `--adopt`.
- Rollout loaders reject interior corruption, conflicting duplicates, inconsistent sample counts and repeated source checkpoints.
- The recheck found and fixed a resume gap: new rollout records now bind their continuation budget and gold answer as well as input tokens. Resume checks the effective token cap; analysis refuses to pool inconsistent rounds.
- Comparisons reject known extraction/rollout hash mismatches and label missing provenance as unverified.
- A CPU audit reconstructs both BOS candidates and compares them with every saved rollout hash.

The recheck ran all ten selected CPU scripts from the delivered ZIP successfully. After the resume fix, all six targeted suites passed; the new guard/audit code has 51 regression tests in total. The final loader accepted all 1,430 cleaned records and rejected the corrupted original. These checks use synthetic fixtures or saved data; they do not validate real GPU execution or the full numerical activation array.

The independent data review reproduced the three main-only recovery AUROCs and bootstrap intervals from saved continuation outcomes, and confirmed every one of the 805 table cells. Research-data probes were not retrained.

New rollout rows use format 4. Older results remain readable for analysis and status, but cannot be resumed as if their missing budget/gold bindings had been recorded. Keep new bound runs in a separate output file.

Use the patched repository and its `JADEN_DATA_FIXES.md` instructions. The old Colab full-extraction wrapper is not updated in Google Drive; its manual BOS adjustment can conflict with future records that already contain exact token IDs.

## What remains open

The folder contains `acts_full.npy` (1,370,173,568 bytes), `meta_full.parquet` and `full_config.json`. **The files are not missing.** The activation array exceeds the 256 MiB download limit available here, so I have not inspected its numerical values, actual header or row binding. The log reports shape `(15380, 29, 1536)`; that is a logged claim, not an independently checked array shape.

I added a read-only CPU check for that array below. It checks shape, row counts, finite values, exact file size and current checksums wherever the file is already accessible. It prints results and never changes the cache. Passing this check still cannot recover missing historical input provenance.

The old full-extraction wrapper reused cached parts by trace ID and layer count; it did not bind them to input-token hashes. If exact historical alignment must be established, the coding team needs to review that provenance gap and decide whether a controlled re-extraction is needed. None has been launched.

The latest saved table completed on its second attempt. Its first attempt failed despite printing `ALL DONE`; the second log and exported tables agree. Table values were inspected, not reproduced by training. Layer sweeps are exploratory, cohorts vary across checkpoints, and separate confidence intervals do not establish paired significance. Detailed evidence is in the handoff's `audit/` folder.

Real Transformers/vLLM execution has not been tested here. The extraction config names the model without pinning an immutable weight revision. Also, `--rebuild` reuses valid same-config parts; it is not an unconditional fresh extraction. These limits remain open despite the passing CPU tests.

In [4]:
# Completed evidence; display only.
import json
SNAPSHOT = json.loads('{"rollout_hash_audit": {"status": "hash_candidates_verified", "original_generation_token_identity": "not_verified", "activation_input_token_identity": "not_verified", "inputs": {"traces_sha256": "c8cb3d8f9b21453b7240e0e024916586faf6e2c1252e8427dc6cacaf1c13a74b", "rollouts_sha256": "0cbd7c1a9b2d27d54b7949e192a76979d740405641ca95a4a769bf706f2cafc4", "generate_code_sha256": "3bb6e97a8dd00ae59e845594cba71849255f2c3efc48137761ca22d3aff6d925", "tokenizer": {"model": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B", "revision": "ad9f0ae0864d7fbcd1cd905e3c6c5b069cc8b562", "files": [{"name": "tokenizer.json", "sha256": "88145e3c3249adc2546ede277e9819d6e405e19072456e4b521cbc724bd60773"}, {"name": "tokenizer_config.json", "sha256": "8ac8c85fb242563c2260baec0909debd69d718af6a0b3d90e6cab62b4d341cd5"}, {"name": "config.json", "sha256": "37bd455e9679d2959536270fed49d25cc7c290a64f6e52abb97c71345a9cee41"}]}}, "valid_audited_records": 1430, "hash_matches_by_bos": {"2": 1430}, "distinct_source_traces": 448, "source_batch_trace_counts": {"first_480_lines": 183, "later_lines": 265}, "grouped_counts": [{"source_batch": "first_480_lines", "checkpoint": "p2048", "matched_bos_count": 2, "records": 183}, {"source_batch": "first_480_lines", "checkpoint": "p4096", "matched_bos_count": 2, "records": 183}, {"source_batch": "first_480_lines", "checkpoint": "p512", "matched_bos_count": 2, "records": 183}, {"source_batch": "later_lines", "checkpoint": "p2048", "matched_bos_count": 2, "records": 265}, {"source_batch": "later_lines", "checkpoint": "p4096", "matched_bos_count": 2, "records": 264}, {"source_batch": "later_lines", "checkpoint": "p512", "matched_bos_count": 2, "records": 265}, {"source_batch": "problem_start", "checkpoint": "t0", "matched_bos_count": 2, "records": 87}], "metadata": {"provided": true, "rows": 11409, "duplicate_trace_positions": 0, "missing_rollout_checkpoints": []}, "issues": [], "interpretation": "Matching candidates cryptographically reproduce saved rollout input hashes using this repository\'s hash definition. They do not independently establish the original generation or historical activation inputs.", "source_batch_note": "Line grouping uses configured first_batch_lines=480; this grouping does not itself prove historical BOS counts."}, "strict_actual_data_loader": {"cleaned_prefixes": 1430, "continuations": 11440, "torn_lines": 0, "checkpoints": {"p512": 448, "p2048": 448, "p4096": 447, "t0": 87}, "raw_corruption_rejected": true, "raw_rejection": "/workspace/scratch/4402c8afbe6e/e0-audit/downloaded/recovery_ceiling:1: malformed JSON inside the rollout file; quarantine explicitly before analysis"}, "focused_cpu_tests": [{"test": "test_cache.py", "exit_code": 0, "seconds": 0.03}, {"test": "test_positions.py", "exit_code": 0, "seconds": 12.42}, {"test": "test_recovery.py", "exit_code": 0, "seconds": 0.43}, {"test": "test_ceiling.py", "exit_code": 0, "seconds": 1.89}, {"test": "test_trace_inputs.py", "exit_code": 0, "seconds": 1.23}, {"test": "test_alignment_guards.py", "exit_code": 0, "seconds": 0.11}, {"test": "test_audit_alignment.py", "exit_code": 0, "seconds": 0.05}, {"test": "test_ceiling_integrity.py", "exit_code": 0, "seconds": 0.92}], "full_metadata_audit": {"status": "metadata_checks_pass", "source_traces": 2202, "metadata_rows": 15380, "layers": [-1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27], "expected_array_payload_bytes": 1370173440, "array_downloaded": false, "numerical_array_validated": false, "array_note": "acts_full.npy was not available locally; metadata and expected byte count do not validate its shape, values, row binding or extraction provenance.", "historical_generation_ids_verified": false, "exact_fixed_prefix_matches": 13219, "fixed_prefix_rows": 13219, "exact_end_prefix_matches": 2161, "end_rows": 2161, "input_hash_overlap": {"rows": 1343, "distinct_source_traces": 448, "matching_metadata_declared_inputs": 794, "mismatching_metadata_declared_inputs": 549, "t0_two_bos_matches": 87, "t0_records": 87, "grouped": [{"source_batch": "main", "checkpoint": "p2048", "status": "declared_input_hash_matches", "rows": 265}, {"source_batch": "main", "checkpoint": "p4096", "status": "declared_input_hash_matches", "rows": 264}, {"source_batch": "main", "checkpoint": "p512", "status": "declared_input_hash_matches", "rows": 265}, {"source_batch": "pilot", "checkpoint": "p2048", "status": "declared_input_hash_mismatch", "rows": 183}, {"source_batch": "pilot", "checkpoint": "p4096", "status": "declared_input_hash_mismatch", "rows": 183}, {"source_batch": "pilot", "checkpoint": "p512", "status": "declared_input_hash_mismatch", "rows": 183}]}, "interpretation": "The overlap check reconstructs the inputs declared by metadata.n_bos and compares their hashes to saved rollout hashes. It does not certify the actual numerical activation inputs. Pilot/main BOS assignments follow full_config and the source line boundary, not original saved generation IDs.", "issues": [], "population": [{"checkpoint": "128", "metadata_n_bos": 1, "rows": 480, "traces": 480, "correct": 213, "wrong": 267, "truncated": 27}, {"checkpoint": "256", "metadata_n_bos": 1, "rows": 480, "traces": 480, "correct": 213, "wrong": 267, "truncated": 27}, {"checkpoint": "512", "metadata_n_bos": 1, "rows": 480, "traces": 480, "correct": 213, "wrong": 267, "truncated": 27}, {"checkpoint": "1024", "metadata_n_bos": 1, "rows": 480, "traces": 480, "correct": 213, "wrong": 267, "truncated": 27}, {"checkpoint": "end", "metadata_n_bos": 1, "rows": 453, "traces": 453, "correct": 213, "wrong": 240, "truncated": 0}, {"checkpoint": "2048", "metadata_n_bos": 1, "rows": 452, "traces": 452, "correct": 185, "wrong": 267, "truncated": 27}, {"checkpoint": "4096", "metadata_n_bos": 1, "rows": 364, "traces": 364, "correct": 117, "wrong": 247, "truncated": 27}, {"checkpoint": "6144", "metadata_n_bos": 1, "rows": 305, "traces": 305, "correct": 79, "wrong": 226, "truncated": 27}, {"checkpoint": "8192", "metadata_n_bos": 1, "rows": 248, "traces": 248, "correct": 45, "wrong": 203, "truncated": 27}, {"checkpoint": "12288", "metadata_n_bos": 1, "rows": 175, "traces": 175, "correct": 11, "wrong": 164, "truncated": 27}, {"checkpoint": "16384", "metadata_n_bos": 1, "rows": 131, "traces": 131, "correct": 5, "wrong": 126, "truncated": 27}, {"checkpoint": "24576", "metadata_n_bos": 1, "rows": 66, "traces": 66, "correct": 0, "wrong": 66, "truncated": 27}, {"checkpoint": "32768", "metadata_n_bos": 1, "rows": 26, "traces": 26, "correct": 0, "wrong": 26, "truncated": 26}, {"checkpoint": "128", "metadata_n_bos": 2, "rows": 1722, "traces": 1722, "correct": 1574, "wrong": 148, "truncated": 14}, {"checkpoint": "256", "metadata_n_bos": 2, "rows": 1722, "traces": 1722, "correct": 1574, "wrong": 148, "truncated": 14}, {"checkpoint": "512", "metadata_n_bos": 2, "rows": 1722, "traces": 1722, "correct": 1574, "wrong": 148, "truncated": 14}, {"checkpoint": "1024", "metadata_n_bos": 2, "rows": 1692, "traces": 1692, "correct": 1544, "wrong": 148, "truncated": 14}, {"checkpoint": "end", "metadata_n_bos": 2, "rows": 1708, "traces": 1708, "correct": 1574, "wrong": 134, "truncated": 0}, {"checkpoint": "2048", "metadata_n_bos": 2, "rows": 1209, "traces": 1209, "correct": 1067, "wrong": 142, "truncated": 14}, {"checkpoint": "4096", "metadata_n_bos": 2, "rows": 606, "traces": 606, "correct": 473, "wrong": 133, "truncated": 14}, {"checkpoint": "6144", "metadata_n_bos": 2, "rows": 355, "traces": 355, "correct": 236, "wrong": 119, "truncated": 14}, {"checkpoint": "8192", "metadata_n_bos": 2, "rows": 253, "traces": 253, "correct": 146, "wrong": 107, "truncated": 14}, {"checkpoint": "12288", "metadata_n_bos": 2, "rows": 134, "traces": 134, "correct": 52, "wrong": 82, "truncated": 14}, {"checkpoint": "16384", "metadata_n_bos": 2, "rows": 74, "traces": 74, "correct": 17, "wrong": 57, "truncated": 14}, {"checkpoint": "24576", "metadata_n_bos": 2, "rows": 30, "traces": 30, "correct": 1, "wrong": 29, "truncated": 14}, {"checkpoint": "32768", "metadata_n_bos": 2, "rows": 13, "traces": 13, "correct": 0, "wrong": 13, "truncated": 13}], "analysis_counts": [{"ckpt": "128", "traces": 599, "problems": 103, "wrong": 243}, {"ckpt": "256", "traces": 599, "problems": 103, "wrong": 243}, {"ckpt": "512", "traces": 599, "problems": 103, "wrong": 243}, {"ckpt": "1024", "traces": 598, "problems": 103, "wrong": 243}, {"ckpt": "end", "traces": 599, "problems": 103, "wrong": 243}, {"ckpt": "2048", "traces": 545, "problems": 97, "wrong": 237}, {"ckpt": "4096", "traces": 448, "problems": 87, "wrong": 212}, {"ckpt": "6144", "traces": 342, "problems": 74, "wrong": 177}, {"ckpt": "8192", "traces": 227, "problems": 59, "wrong": 128}, {"ckpt": "12288", "traces": 78, "problems": 26, "wrong": 46}, {"ckpt": "16384", "traces": 26, "problems": 9, "wrong": 15}, {"ckpt": "24576", "traces": 2, "problems": 1, "wrong": 1}, {"ckpt": "32768", "traces": 0, "problems": 0, "wrong": 0}], "table_population_counts_checked": true, "table_population_counts_match": true, "hashes": {"traces": "c8cb3d8f9b21453b7240e0e024916586faf6e2c1252e8427dc6cacaf1c13a74b", "meta_full": "1addd8bd813ae99747198fa90edb1acd03f37ad0833f8d086875ec983faa48e3", "full_config": "b5aebe2a9bbd06242d6faf9b4d587f62c2a5386eae50d74e29e4ee5b85c9d614", "rollouts": "0cbd7c1a9b2d27d54b7949e192a76979d740405641ca95a4a769bf706f2cafc4", "generation_code": "3bb6e97a8dd00ae59e845594cba71849255f2c3efc48137761ca22d3aff6d925"}, "tokenizer": {"model": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B", "revision": "ad9f0ae0864d7fbcd1cd905e3c6c5b069cc8b562", "files": [{"name": "tokenizer.json", "sha256": "88145e3c3249adc2546ede277e9819d6e405e19072456e4b521cbc724bd60773"}, {"name": "tokenizer_config.json", "sha256": "8ac8c85fb242563c2260baec0909debd69d718af6a0b3d90e6cab62b4d341cd5"}, {"name": "config.json", "sha256": "37bd455e9679d2959536270fed49d25cc7c290a64f6e52abb97c71345a9cee41"}]}}, "full_table_review": {"completion": {"extraction_log": "run.log records assembly of 15380 rows, shape (15380,29,1536), after 2202/2202 traces", "initial_table_attempt": "run.log records AssertionError because one main-run ceiling trace lacks rollouts at 4096; ALL DONE subsequently printed despite error", "rerun": "run_table.log excludes trace 216_2 at 4096, reaches end, and prints both complete tables and all checkpoint counts; no traceback in rerun log", "rerun_has_traceback": false, "csv_log_agreement": {"e0_full_table.csv": {"csv_rows": 62, "rows_matching_log": 62, "data_checkpoint_columns": 10}, "e0_full_table_ceiling.csv": {"csv_rows": 66, "rows_matching_log": 66, "data_checkpoint_columns": 3}}, "interpretation": "Saved result tables are consistent with a completed rerun; this does not independently reproduce numerical probe values or certify activation provenance", "long_to_wide_validation": {"long_rows": 805, "unique_result_keys": 805, "wide_numeric_cells_checked": 805, "matching_rounding": 805, "population_footer_checks": 796, "missing_or_extra_result_cells": 0}}, "ceiling_population_by_checkpoint": [{"ckpt": "512", "traces": 265, "problems": 52, "pairs": 268}, {"ckpt": "2048", "traces": 265, "problems": 52, "pairs": 268}, {"ckpt": "4096", "traces": 264, "problems": 52, "pairs": 267, "missing_rollout_trace": "216_2"}], "not_verified": {"acts_full_npy": "1.37 GB activation file was not downloaded because the connected download tool has a 256 MiB limit; shape is reported by config/log only, values and file hash not inspected", "numerical_activation_prefix_equality": "Not rerun; no GPU/model execution", "original_generation_token_ids": "Not recorded in historical source; one-/two-BOS mapping is inference", "numerical_probe_reproduction": "No probe runs; reviewed saved CSV/log agreement only", "metadata_alignment": "Delegated to separate audit; not claimed by this table-only review"}}, "drive_readonly_inventory": {"folder_url": "https://drive.google.com/drive/folders/1GgqDKOUxHDvuj9YH3Ln1TEV-1hicCli7", "read_only": true, "downloaded": [{"name": "meta_full.parquet", "id": "12ojMl0WfFNC3-5DiOqzvdGMeGducvwwf"}, {"name": "full_config.json", "id": "1k85TPWOXGeZk4rEUhykAJCqbzlmeE7hK"}, {"name": "e0_full_table.csv", "id": "1r2AuWq-3vNIXisR2Ar5WEmrKXzBT1MvB"}, {"name": "e0_full_table_counts.csv", "id": "1F3ImGNTop7ygdUEmYHg5V0huhhJAdFO5"}, {"name": "e0_full_table_ceiling.csv", "id": "123Q3AF-c4GMQNrAeEyx-OaOgB_c5x8wj"}, {"name": "run.log", "id": "1jmP-Sc74YUIknzn6iRcBUxX3Smg4PgnY"}, {"name": "run_table.log", "id": "1oX3RQf9VqM7empgJWM3uLAIeKRo98EQQ"}], "activation_file": {"name": "acts_full.npy", "id": "1CAYA1G_qH115Hk3iT_k5b6c03kcuHMjo", "size_bytes": 1370173568, "download_status": "blocked_by_268435456_byte_download_limit", "numerical_integrity": "not_checked"}, "historical_provenance": "No recorded activation input hashes in full_config or metadata"}, "full_cache_available_files_audit": {"cache_dir": "/workspace/scratch/4402c8afbe6e/e0-audit/drive-e0-full", "checks": {"acts_full.npy.exists": {"passed": false}, "config.layer_convention": {"passed": true}, "config.layer_order": {"detail": {"actual": [-1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27], "expected": [-1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27]}, "passed": true}, "config.readable": {"passed": true}, "config.rows_match_metadata": {"detail": {"configured": 15380, "observed": 15380}, "passed": true}, "config.traces_match_metadata": {"detail": {"configured": 2202, "observed": 2202}, "passed": true}, "full_config.json.exists": {"passed": true}, "full_config.json.unchanged_during_audit": {"passed": true}, "meta_full.parquet.exists": {"passed": true}, "meta_full.parquet.unchanged_during_audit": {"passed": true}, "metadata.checkpoints_in_config": {"detail": {"unknown": []}, "passed": true}, "metadata.fixed_checkpoint_matches_position": {"passed": true}, "metadata.nonempty": {"passed": true}, "metadata.nonnull_trace_checkpoint": {"passed": true}, "metadata.positive_integer_positions": {"passed": true}, "metadata.readable": {"passed": true}, "metadata.required_columns": {"detail": {"missing": []}, "passed": true}, "metadata.sequential_rows": {"passed": true}, "metadata.unique_trace_checkpoint": {"passed": true}}, "files": {"full_config.json": {"bytes": 460, "sha256_observed_now": "b5aebe2a9bbd06242d6faf9b4d587f62c2a5386eae50d74e29e4ee5b85c9d614"}, "meta_full.parquet": {"bytes": 27214377, "sha256_observed_now": "1addd8bd813ae99747198fa90edb1acd03f37ad0833f8d086875ec983faa48e3"}}, "historical_input_provenance": "unverified", "limitations": ["Matching file size, shape and fresh checksums do not prove historical token inputs.", "New checksums describe the files read now; no prior extraction manifest is created or inferred.", "This audit does not compare activations with a fresh model pass or rerun probe experiments."], "read_only": true, "scope": "structural_only", "structural_checks_passed": false}, "independent_recheck": {"review_passes": ["Independent reconstruction from actual source, metadata, tokenizer and saved rollout outcomes", "Independent code review plus CPU regression/integration tests; confirmed bug corrected", "Notebook execution, ZIP integrity, source/archive equality and packaged-file checks"], "data_findings_confirmed": true, "confirmed_bug_fixed": "Resume could reuse completed results after a per-prefix continuation budget or gold answer changed.", "fix": "New rollout rows bind input tokens, continuation budget and gold answer; resume checks effective caps and analysis rejects inconsistent pooling.", "compatibility": "Legacy rows remain readable, but cannot be resumed as newly verified row-format-4 results. Use a separate output for new bound runs.", "original_delivered_cpu_scripts_passed": 10, "post_fix_targeted_cpu_scripts_passed": 6, "new_regression_test_count": 51, "shared_drive_modified": false, "limitations": ["The real 1.37 GB acts_full.npy exceeds the download limit and was not numerically inspected.", "Historical activation input provenance and original generation token IDs cannot be certified from metadata or newly computed hashes.", "Real Transformers/vLLM model execution was not tested; CPU suites use synthetic fixtures and stubs where necessary.", "Research-data probes were not retrained; existing probe AUROCs were cross-checked against exports/logs, not independently reproduced.", "The extraction config identifies the model by name, not an immutable model-weight revision.", "--rebuild only replaces incompatible/unmanifested caches and is not an unconditional fresh forward pass."]}, "independent_data_recheck": {"independent_of_previous_audit_helpers": true, "source_rows": 2202, "source_problems": 367, "metadata_rows": 15380, "fixed_prefix_rows_compared": 13219, "end_rows_compared": 2161, "token_count_flag_rule": "n_gen minus retokenized length differs from 1 for completed traces or 0 for truncated traces; heuristic only", "source_trace_residual_count_flags": 38, "rollout_records": 1430, "sample_count_histogram": {"8": 1430}, "hash_candidate_counts": {"(2,)": 1430}, "overlap_counts": {"(\'main\', \'p4096\', True)": 264, "(\'pilot\', \'p4096\', False)": 183, "(\'main\', \'p512\', True)": 265, "(\'main\', \'p2048\', True)": 265, "(\'pilot\', \'p2048\', False)": 183, "(\'pilot\', \'p512\', False)": 183}, "source_rollout_traces": 448, "source_rollout_residual_flags": 16, "full_grid_population_counts_match": true, "main_rollout_population": [{"ckpt": "512", "traces": 265, "problems": 52, "pairs": 268, "auroc": 0.6511194029850746}, {"ckpt": "2048", "traces": 265, "problems": 52, "pairs": 268, "auroc": 0.6399253731343284}, {"ckpt": "4096", "traces": 264, "problems": 52, "pairs": 267, "auroc": 0.7902621722846442}], "main_rhat_direct_recomputation": [{"ckpt": "512", "max_error": 0.0}, {"ckpt": "2048", "max_error": 0.0}, {"ckpt": "4096", "max_error": 0.0}], "wide_cells_checked": 805, "issues": [], "activation_array_checked": false}, "final_loader_recheck": {"cleaned_records": 1430, "continuations": 11440, "torn_lines": 0, "raw_corruption_rejected": true, "raw_rejection": "/workspace/scratch/4402c8afbe6e/e0-audit/downloaded/recovery_ceiling:1: malformed JSON inside the rollout file; quarantine explicitly before analysis"}}')
m = SNAPSHOT["full_metadata_audit"]
print("Full metadata rows checked:", m["metadata_rows"])
print("Fixed prefixes matched:", m["exact_fixed_prefix_matches"], "/", m["fixed_prefix_rows"])
print("End prefixes matched:", m["exact_end_prefix_matches"], "/", m["end_rows"])
print("Main declared input matches:", m["input_hash_overlap"]["matching_metadata_declared_inputs"])
print("Pilot declared input mismatches:", m["input_hash_overlap"]["mismatching_metadata_declared_inputs"])
print("Full activation array numerically checked:", m["numerical_array_validated"])

Full metadata rows checked: 15380
Fixed prefixes matched: 13219 / 13219
End prefixes matched: 2161 / 2161
Main declared input matches: 794
Pilot declared input mismatches: 549
Full activation array numerically checked: False


## Optional CPU checks

Leave both switches **False** to display the report only. No model or GPU is needed.

- **Full cache check:** unzip `Jaden_Repo_Fixes.zip` and point `CACHE_DIR` at the existing `e0_full` folder. Set `RUN_FULL_CACHE_CHECK = True`. This reads the files and prints results; it makes no changes to Drive or the cache. The folder must already be accessible in the runtime. Dependencies: `numpy pandas pyarrow`.
- **Repeat the rollout hash audit:** supply the original `traces_B.jsonl`, unzip `Jaden_E0_Handoff`, and set `RUN_CPU_AUDIT = True`. This writes a new report under `/content`, outside the input folders. Additional dependencies: `tokenizers jinja2`.

These checks can verify structure or reconstructed hash candidates. They do not certify undocumented historical model inputs.

In [6]:
from pathlib import Path
import subprocess, sys, datetime

RUN_FULL_CACHE_CHECK = False
RUN_CPU_AUDIT = False
REPO = Path("/content/Ghait---MJJS-jaden-data-fixes")
CACHE_DIR = Path("/content/drive/MyDrive/e0_probe_cmp/e0_full")  # Adjust if mounted elsewhere.
HANDOFF = Path("/content/Jaden_E0_Handoff")
TRACES = Path("/content/e0_raw/traces_B.jsonl")

if RUN_FULL_CACHE_CHECK:
    subprocess.run([
        sys.executable, "-B", str(REPO / "src/audit_full_cache.py"),
        "--cache-dir", str(CACHE_DIR),
        "--expected-acts-bytes", "1370173568",
    ], check=True)

if RUN_CPU_AUDIT:
    output = Path("/content") / ("jaden_audit_" + datetime.datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
    subprocess.run([
        sys.executable, "-B", str(REPO / "src/audit_alignment.py"),
        "--generation-code", str(REPO / "src/generate.py"),
        "--traces", str(TRACES),
        "--rollouts", str(HANDOFF / "cleaned_data/recovery_ceiling_clean.jsonl"),
        "--rollout-manifest", str(HANDOFF / "cleaned_data/cleaning_manifest.json"),
        "--tokenizer-dir", str(HANDOFF / "tokenizer"),
        "--out", str(output),
    ], check=True)
    print("Audit finished. Read its findings and limitations in", output)

if not RUN_FULL_CACHE_CHECK and not RUN_CPU_AUDIT:
    print("No rerun requested. Saved findings are shown above.")

No rerun requested. Saved findings are shown above.
